# 10. Busca de copy-move como feature do classificador

A técnica 1 (notebook 07: busca cega de copy-move com verificação do resíduo de ruído) foi avaliada como localizador de regiões. Este notebook testa se ela também ajuda a **classificar a imagem inteira**: uma feature binária entra no classificador v2, e a validação cruzada diz se a AUC melhora.

**Feature nova (binária, única):** vale 1 se a imagem tem algum par aprovado pela técnica 1 com a configuração congelada em `results/copy_move_residuo_val.json` (piso de intensidade 0,95, mediana 3 x 3, limiar do resíduo 0,9, deslocamento de até 2 px, `MIN_SIDE` 6), e 0 caso contrário. A cobertura mínima da técnica 1 mede regiões anotadas e não entra na feature de imagem. Nenhuma anotação é usada.

**Dados e protocolo:** as mesmas 769 imagens do classificador v2 (train + val do *Find it again!*, 127 adulteradas e 642 autênticas) e as mesmas 15 partições (validação cruzada estratificada 5 x 3, seed 42), com padronização e regressão logística balanceada. O split `test` não é aberto. O cálculo é sequencial, sem multiprocessing.

O protocolo, o critério e os hashes do código foram gravados em `results/classificador_copy_move_cv_fixado.json` antes de qualquer execução sobre imagens. **Critério:** a AUC externa média do candidato (v2 + a feature) deve ser pelo menos 0,70 e o limite inferior do IC 95% corrigido (Nadeau e Bengio) da diferença pareada contra o v2 deve ser maior que zero.

**Ressalva:** a configuração da técnica 1 (piso, k, limiar) foi escolhida no `val` do *Find it again!*, que faz parte dos 769. A feature carrega, portanto, uma seleção feita em parte dos dados da avaliação. Os números aparecem apenas como saída das células; este notebook não registra conclusões.

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from classification import bootstrap_auc, cv_evaluate
from comparacao_cv import auc_por_particao, diferenca_pareada
from copy_move_feature import feature_lote
from dataset_loader import load_dataset
from features import FEATURE_NAMES, load_or_extract
from features_v2 import V2_NAMES, load_or_extract_v2

RAIZ = Path("..")
DATASET_ROOT = str(RAIZ / "data/findit2")

## 1. Protocolo congelado

A célula confere que o código novo e os arquivos reutilizados sem alteração (`src/copy_move_residuo.py`, `src/copy_move.py`, `src/comparacao_cv.py`) ainda têm os hashes registrados, e que o arquivo de configuração da técnica 1 é o mesmo.

In [2]:
fixado = json.loads((RAIZ / "results/classificador_copy_move_cv_fixado.json").read_text(encoding="utf-8"))
print("congelado em:", fixado["congelado_em"])
esperados = {**fixado["codigo"], **fixado["reutiliza_sem_alterar"]}
for arquivo, esperado in esperados.items():
    print(arquivo, "idêntico ao congelado:", hashlib.sha256((RAIZ / arquivo).read_bytes()).hexdigest() == esperado)
caminho_config = "results/copy_move_residuo_val.json"
print(caminho_config, "idêntico ao congelado:", hashlib.sha256((RAIZ / caminho_config).read_bytes()).hexdigest() == fixado["feature_nova"]["sha256_arquivo_da_configuracao"])
config_tecnica = fixado["feature_nova"]["configuracao"]
print({k: config_tecnica[k] for k in ("piso_intensidade", "k", "limiar_residuo")})

congelado em: 2026-11-06T17:57:08
src/copy_move_feature.py idêntico ao congelado: True
src/copy_move_residuo.py idêntico ao congelado: True
src/copy_move.py idêntico ao congelado: True
src/comparacao_cv.py idêntico ao congelado: True
results/copy_move_residuo_val.json idêntico ao congelado: True
{'piso_intensidade': 0.95, 'k': 3, 'limiar_residuo': 0.9}


## 2. Etapa 1: a feature nas 769 imagens

A busca de copy-move da técnica 1 roda, imagem a imagem (sem paralelismo), em todas as imagens de train e val. A feature é binária e está sempre definida; qualquer falha interrompe a execução. O resultado é guardado em `data/features_copy_move_cv.csv` (fora do Git). Reporta-se a fração de imagens com a feature igual a 1 em cada classe.

In [3]:
amostras = {split: load_dataset(DATASET_ROOT, split) for split in ("train", "val")}
inicio = time.time()
feature = feature_lote([s.image_path for S in amostras.values() for s in S],
                       config_tecnica["piso_intensidade"], config_tecnica["k"], config_tecnica["limiar_residuo"])
print("tempo: %.1f min | imagens: %d" % ((time.time() - inicio) / 60, len(feature)))
tabela = pd.DataFrame([{"image": s.image_path.name, "is_tampered": int(s.is_tampered), "copy_move_par": feature[str(s.image_path)]}
                       for S in amostras.values() for s in S])
tabela.to_csv(RAIZ / "data/features_copy_move_cv.csv", index=False)
print("imagens:", len(tabela), "| adulteradas:", int(tabela["is_tampered"].sum()), "| autênticas:", int((tabela["is_tampered"] == 0).sum()))
print("feature indefinida:", int(tabela["copy_move_par"].isna().sum()), "| valores distintos:", sorted(tabela["copy_move_par"].unique()))
resumo = tabela.groupby("is_tampered")["copy_move_par"].agg(["sum", "count", "mean"]).rename(index={0: "autênticas", 1: "adulteradas"})
print(resumo.rename(columns={"sum": "com par", "count": "imagens", "mean": "fração"}).round(4).to_string())

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)
aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)


tempo: 0.8 min | imagens: 769
imagens: 769 | adulteradas: 127 | autênticas: 642
feature indefinida: 0 | valores distintos: [np.int64(0), np.int64(1)]
             com par  imagens  fração
is_tampered                          
autênticas       135      642  0.2103
adulteradas       70      127  0.5512


## 3. Etapa 2: avaliação na validação cruzada

Reprodução do protocolo do v2: validação cruzada estratificada, 5 folds x 3 repetições, seed 42, com padronização e regressão logística balanceada (`class_weight="balanced"`, `max_iter=1000`). Como `cv_evaluate` devolve só médias, `auc_por_particao` repete o divisor e a semente para obter a AUC de cada partição. A célula confere que a média reproduz a AUC externa registrada em `results/configuracao_congelada.json` (0,665416 com desvio 0,046784).

In [4]:
v1 = {split: load_or_extract(S, str(RAIZ / f"data/features_{split}.csv")) for split, S in amostras.items()}
v2 = {split: load_or_extract_v2(S, str(RAIZ / f"data/features_v2_{split}.csv")) for split, S in amostras.items()}
dados = pd.concat([v1[split].merge(v2[split], on="image", validate="one_to_one") for split in amostras], ignore_index=True)
dados = dados.merge(tabela[["image", "copy_move_par"]], on="image", validate="one_to_one")
NOVA = ["copy_move_par"]
y = dados["is_tampered"].to_numpy()
print("train + val:", len(dados), "| adulteradas:", int(y.sum()), "| autênticas:", int((y == 0).sum()))
modelo = lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
colunas_v2 = FEATURE_NAMES + V2_NAMES
colunas_candidato = colunas_v2 + NOVA
config = json.loads((RAIZ / "results/configuracao_congelada.json").read_text(encoding="utf-8"))
auc_v2 = auc_por_particao(modelo, dados[colunas_v2].to_numpy(), y)
resumo_v2 = cv_evaluate(modelo, dados[colunas_v2].to_numpy(), y)
print("v2: média das partições = %.6f | desvio = %.6f | cv_evaluate: %.6f +- %.6f | registrado: %.6f +- %.6f" % (
    auc_v2.mean(), auc_v2.std(), resumo_v2["auc_mean"], resumo_v2["auc_std"],
    config["auc_externa_cv"]["media"], config["auc_externa_cv"]["desvio"]))
reproduz = bool(abs(auc_v2.mean() - config["auc_externa_cv"]["media"]) < 1e-9)
print("reproduz o registrado:", reproduz)

train + val: 769 | adulteradas: 127 | autênticas: 642
v2: média das partições = 0.665416 | desvio = 0.046784 | cv_evaluate: 0.665416 +- 0.046784 | registrado: 0.665416 +- 0.046784
reproduz o registrado: True


Comparação lado a lado, nas mesmas 15 partições. A diferença pareada é AUC(candidato) menos AUC(v2) em cada partição. O IC 95% usa a correção de Nadeau e Bengio para validação cruzada repetida (as partições compartilham dados); o IC ingênuo é mostrado só para comparação.

In [5]:
auc_cand = auc_por_particao(modelo, dados[colunas_candidato].to_numpy(), y)
comparacao = diferenca_pareada(auc_cand, auc_v2)
por_particao = pd.DataFrame({"v2": auc_v2, "candidato": auc_cand, "diferenca": auc_cand - auc_v2})
print(por_particao.round(4).to_string())
print("v2:        AUC = %.4f +- %.4f" % (auc_v2.mean(), auc_v2.std()))
print("candidato: AUC = %.4f +- %.4f" % (auc_cand.mean(), auc_cand.std()))
print("diferença pareada média = %.4f (desvio entre partições %.4f)" % (comparacao["diferenca_media"], comparacao["desvio"]))
print("IC 95%% corrigido (Nadeau e Bengio): [%.4f, %.4f]" % tuple(comparacao["ic95_corrigido"]))
print("IC 95%% ingênuo: [%.4f, %.4f]" % tuple(comparacao["ic95_ingenuo"]))
print("partições em que o candidato supera o v2:", int((auc_cand > auc_v2).sum()), "de", len(auc_v2))

        v2  candidato  diferenca
0   0.6412     0.7620     0.1208
1   0.6983     0.7593     0.0610
2   0.6195     0.6670     0.0474
3   0.6695     0.7203     0.0509
4   0.6909     0.7188     0.0278
5   0.7374     0.7828     0.0454
6   0.6581     0.7398     0.0817
7   0.6028     0.6716     0.0688
8   0.7681     0.8078     0.0397
9   0.5925     0.6597     0.0672
10  0.6623     0.7052     0.0430
11  0.7058     0.7260     0.0201
12  0.6623     0.7309     0.0685
13  0.6322     0.7073     0.0750
14  0.6403     0.7550     0.1147
v2:        AUC = 0.6654 +- 0.0468
candidato: AUC = 0.7276 +- 0.0409
diferença pareada média = 0.0621 (desvio entre partições 0.0284)
IC 95% corrigido (Nadeau e Bengio): [0.0278, 0.0964]
IC 95% ingênuo: [0.0464, 0.0779]
partições em que o candidato supera o v2: 15 de 15


AUC isolada da feature nova (adulterada contra autêntica, nas 769 imagens), com IC de bootstrap sobre as imagens (2000 reamostragens, seed 42). Para uma feature binária, essa AUC é (sensibilidade + especificidade) / 2 da regra "tem par". Uma AUC de 0,5 é o acaso.

In [6]:
valor = dados[NOVA[0]].to_numpy()
isolada = {"auc": float(roc_auc_score(y, valor)), "ic_bootstrap": list(bootstrap_auc(y, valor)),
           "sensibilidade": float(valor[y == 1].mean()), "taxa_falso_alarme": float(valor[y == 0].mean())}
print("AUC isolada = %.3f, IC bootstrap [%.3f, %.3f]" % (isolada["auc"], *isolada["ic_bootstrap"]))
print("sensibilidade da regra 'tem par': %.1f%% | falso alarme: %.1f%%" % (100 * isolada["sensibilidade"], 100 * isolada["taxa_falso_alarme"]))

AUC isolada = 0.670, IC bootstrap [0.623, 0.716]
sensibilidade da regra 'tem par': 55.1% | falso alarme: 21.0%


Critério congelado: o candidato vence se a AUC externa média na validação cruzada for pelo menos 0,70 **e** o limite inferior do IC 95% corrigido da diferença pareada contra o v2 for maior que zero.

In [7]:
atingiu_auc = bool(auc_cand.mean() >= 0.70)
atingiu_diferenca = bool(comparacao["ic95_corrigido"][0] > 0)
print("AUC externa do candidato >= 0,70:", atingiu_auc, "(%.4f)" % auc_cand.mean())
print("limite inferior do IC95 corrigido da diferença > 0:", atingiu_diferenca, "(%.4f)" % comparacao["ic95_corrigido"][0])
print("critério atingido:", atingiu_auc and atingiu_diferenca)
saida = {"escopo": "train + val do Find it again!; test não aberto", "n_imagens": int(len(dados)), "feature_nova": NOVA,
         "configuracao_tecnica_1": config_tecnica,
         "fracao_com_par": {"adulteradas": float(valor[y == 1].mean()), "autenticas": float(valor[y == 0].mean()),
                            "n_adulteradas_com_par": int(valor[y == 1].sum()), "n_autenticas_com_par": int(valor[y == 0].sum())},
         "v2": {"auc_media": float(auc_v2.mean()), "auc_desvio": float(auc_v2.std()), "auc_por_particao": [float(v) for v in auc_v2], "reproduz_registrado": reproduz},
         "candidato": {"auc_media": float(auc_cand.mean()), "auc_desvio": float(auc_cand.std()), "auc_por_particao": [float(v) for v in auc_cand]},
         "diferenca_pareada": comparacao, "auc_isolada": isolada,
         "criterio": {"auc_externa_minima": 0.70, "atingiu_auc": atingiu_auc, "atingiu_diferenca": atingiu_diferenca, "atingido": atingiu_auc and atingiu_diferenca},
         "codigo_sha256": fixado["codigo"]}
(RAIZ / "results/classificador_copy_move_cv.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

AUC externa do candidato >= 0,70: True (0.7276)
limite inferior do IC95 corrigido da diferença > 0: True (0.0278)
critério atingido: True
gravado
